# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mkhlor006/Flyrank_internship_ML/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

**Unit of analysis:** One row represents one content page for one client on one report date.

**Time window:** I use the March 2026 partition of `fact_content_daily_performance`, covering 1 March 2026 to 31 March 2026.

The data is therefore at a daily content-page level: one `client_hash_id` + one `content_hash_id` + one `report_date`.


In [5]:
import pandas as pd

march_path = (
    "hf://datasets/FlyRank/internship-warehouse/"
    "fact_content_daily_performance/month=2026-03"
)

march = pd.read_parquet(march_path)

print("Rows:", len(march))
print("Columns:", len(march.columns))
print(
    "Date range:",
    march["report_date"].min(),
    "to",
    march["report_date"].max()
)

Rows: 9841378
Columns: 30
Date range: 2026-03-01 to 2026-03-31


In [6]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

print("Rows:", len(march))
print("Columns:", len(march.columns))
print("Date range:", march["report_date"].min(), "to", march["report_date"].max())

Rows: 9841378
Columns: 30
Date range: 2026-03-01 to 2026-03-31


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

**Features**

I would use the following five observable features:

1. `gsc_impressions` — shows how much search visibility the page received.
2. `gsc_clicks` — shows observed search clicks.
3. `gsc_avg_position` — shows the page's observed average search position.
4. `ga4_pageviews` — shows observed page views.
5. `ga4_sessions` — shows observed sessions.

These are measurable signals available in the warehouse before a future content-prioritisation decision.

**Label / proxy**

For the eventual ML task, the target would be a future decline or performance-change label defined from a later observation window. The target must be based on a future outcome rather than the same-period features.

**Context**

`client_hash_id`, `content_hash_id`, `report_date`, and `month` provide identity and time context. They help group and split the data but should not be used as predictive features.

**Excluded**

I deliberately exclude `trend_direction` and `trend_pct` when they are available because they are derived from the outcome and would leak the answer into the features. I also exclude client and content IDs as predictive features because they are identifiers rather than meaningful behavioural signals.


In [7]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.



## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [8]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

duplicates = march.duplicated(
    subset=["client_hash_id", "content_hash_id", "report_date"]
).sum()

print("Duplicate rows at client + content + date grain:", duplicates)

Duplicate rows at client + content + date grain: 0


In [9]:
print("Row count:", len(march))
print("Earliest date:", march["report_date"].min())
print("Latest date:", march["report_date"].max())

Row count: 9841378
Earliest date: 2026-03-01
Latest date: 2026-03-31


In [10]:
available = march[
    (march["gsc_data_available"] == True) &
    (march["ga4_data_available"] == True)
]

print("Rows with both GSC and GA4 available:", len(available))
print(
    "Percentage retained:",
    round(len(available) / len(march) * 100, 2),
    "%"
)

Rows with both GSC and GA4 available: 364347
Percentage retained: 3.7 %


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

## Data limits

The March 2026 slice provides measured search and analytics performance, but it cannot tell me why a page's performance changed. For example, it cannot by itself establish that a Google algorithm update, competitor activity, content quality, or a particular content change caused a performance change.

Data availability is another limitation. In the March slice, 364,347 of 9,841,378 rows (3.7%) had both GSC and GA4 data available. Therefore, analyses requiring both sources would use a much smaller subset of the data.

The data also has different history availability across clients. Results should therefore be treated as observed and directional decision-support evidence rather than causal proof.

### Five features

I selected five features that represent observable search and site-performance signals:

| Feature            | Why it is knowable at the decision moment                                           |
| ------------------ | ----------------------------------------------------------------------------------- |
| `gsc_impressions`  | Knowable because Search Console records the page's observed search impressions.     |
| `gsc_clicks`       | Knowable because Search Console records observed clicks for the page.               |
| `gsc_avg_position` | Knowable because the warehouse contains the observed average search position.       |
| `ga4_pageviews`    | Knowable when GA4 data is available because the page's observed views are recorded. |
| `ga4_sessions`     | Knowable when GA4 data is available because observed sessions are recorded.         |


In [11]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
feature_cols = [
    "client_hash_id",
    "content_hash_id",
    "report_date",
    "gsc_impressions",
    "gsc_clicks",
    "gsc_avg_position",
    "ga4_pageviews",
    "ga4_sessions"
]

feature_frame = march[feature_cols].copy()

print("Feature frame shape:", feature_frame.shape)

display(feature_frame.head())

Feature frame shape: (9841378, 8)


,client_hash_id,content_hash_id,report_date,gsc_impressions,gsc_clicks,gsc_avg_position,ga4_pageviews,ga4_sessions
0,client_73cda7b4e4f265ea,content_b7e512995f79d5a6,2026-03-01,20,0,3.350000,NaN,NaN
1,client_73cda7b4e4f265ea,content_05597932fe4da067,2026-03-01,1,0,0.000000,NaN,NaN
2,client_73cda7b4e4f265ea,content_7a105f548d9c6916,2026-03-01,125,1,4.928000,NaN,NaN
3,client_73cda7b4e4f265ea,content_905aa32a0230694e,2026-03-01,7,0,4.000000,NaN,NaN
4,client_73cda7b4e4f265ea,content_a3ea9792f793ec72,2026-03-01,11,0,2.272727,NaN,NaN


### The leakage trap

I deliberately test what happens when information derived from the outcome is included as a feature. If the feature contains the answer that the model is supposed to predict, performance can become unrealistically strong. This is leakage because that information would not be legitimately available at the decision moment. After demonstrating the effect, the leaked feature is removed and only legitimate pre-decision features are retained.


## Leakage trap

To demonstrate leakage, I create a simple outcome from later observations and then deliberately include that outcome as a feature. This should make the model appear unusually strong because the feature contains information derived from the answer.

This is not a valid modelling setup. The leaked feature would not be legitimately available at the decision moment. After demonstrating the effect, I remove the leaked feature and retain only features that could have been known when the decision was made.

In [12]:
import pandas as pd
import numpy as np
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score

# Sort observations by client, content and date
march_sorted = march.sort_values(
    ["client_hash_id", "content_hash_id", "report_date"]
).copy()

# Define a future outcome:
# whether the page gets at least one click on its next observed day
march_sorted["future_click"] = (
    march_sorted.groupby(
        ["client_hash_id", "content_hash_id"]
    )["gsc_clicks"]
    .shift(-1)
    .fillna(0)
    > 0
).astype(int)

print("Target distribution:")
print(march_sorted["future_click"].value_counts())

Target distribution:
future_click
0    9436019
1     405359
Name: count, dtype: int64


In [13]:
from sklearn.model_selection import train_test_split

honest_features = [
    "gsc_impressions",
    "gsc_clicks",
    "gsc_avg_position",
    "ga4_pageviews",
    "ga4_sessions"
]

leak_features = honest_features + ["future_click"]

model_data = march_sorted[
    honest_features + ["future_click"]
].copy()

model_data = model_data.replace(
    [np.inf, -np.inf], np.nan
).fillna(0)

X_leak = model_data[leak_features]
y_leak = model_data["future_click"]

X_train, X_test, y_train, y_test = train_test_split(
    X_leak,
    y_leak,
    test_size=0.2,
    random_state=42,
    stratify=y_leak
)

leaky_tree = DecisionTreeClassifier(
    max_depth=3,
    random_state=42
)

leaky_tree.fit(X_train, y_train)

leaky_pred = leaky_tree.predict(X_test)

print(
    "Leaky model accuracy:",
    round(accuracy_score(y_test, leaky_pred), 3)
)

Leaky model accuracy: 1.0


In [14]:
X_honest = model_data[honest_features]

X_train, X_test, y_train, y_test = train_test_split(
    X_honest,
    y_leak,
    test_size=0.2,
    random_state=42,
    stratify=y_leak
)

honest_tree = DecisionTreeClassifier(
    max_depth=3,
    random_state=42
)

honest_tree.fit(X_train, y_train)

honest_pred = honest_tree.predict(X_test)

print(
    "Honest model accuracy:",
    round(accuracy_score(y_test, honest_pred), 3)
)

Honest model accuracy: 0.965


### Leakage verdict

The deliberately leaked model uses `future_click`, which is derived from the outcome being predicted. Its performance can therefore be misleadingly strong. I would not use this feature in a real model because it would not be available at the decision moment.

The honest model removes `future_click` and uses only the five selected observable features. The honest result is the number I would retain for further experimentation.

This demonstrates that a strong score is not enough: the features must also represent information that would genuinely be available before the decision.

### Leakage verdict

The target contains 9,436,019 observations labelled 0 and 405,359 labelled 1. When I deliberately included `future_click`, which is derived from the outcome being predicted, the model achieved 1.000 accuracy. This is an example of data leakage because the feature contains the answer and would not legitimately be available at the decision moment.

After removing the leaked feature and using only the five observable features, the accuracy fell to 0.965. I keep the honest result and discard the leaky feature. The difference shows why a very strong score should not be trusted unless the features are available before the prediction decision.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.